In [20]:
spark = SparkSession.builder.appName('Ass2').getOrCreate()

In [21]:
from pyspark.sql.functions import col, timestamp_seconds

ratings_raw_df = spark.read.format('csv').option('Header', True).option('InferSchema', True).load('/Ass2/Input/ratings.csv')
movies_df = spark.read.option('Header',True).option('InferSchema', True).csv('/Ass2/Input/movies.csv')
tags_raw_df = spark.read.format('csv').options(Header='True', InferSchema ='True').load('/Ass2/Input/tags.csv')

ratings_df = ratings_raw_df.withColumn("timestamp", timestamp_seconds(col('timestamp')))
tags_df = tags_raw_df.withColumn("timestamp", timestamp_seconds(col('timestamp')))

In [99]:
movies_df.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



In [50]:
tags_df.show(5)

+------+-------+---------------+-------------------+
|userId|movieId|            tag|          timestamp|
+------+-------+---------------+-------------------+
|     2|  60756|          funny|2015-10-24 19:29:54|
|     2|  60756|Highly quotable|2015-10-24 19:29:56|
|     2|  60756|   will ferrell|2015-10-24 19:29:52|
|     2|  89774|   Boxing story|2015-10-24 19:33:27|
|     2|  89774|            MMA|2015-10-24 19:33:20|
+------+-------+---------------+-------------------+
only showing top 5 rows



In [22]:
ratings_df.show(10)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-30 18:45:03|
|     1|      3|   4.0|2000-07-30 18:20:47|
|     1|      6|   4.0|2000-07-30 18:37:04|
|     1|     47|   5.0|2000-07-30 19:03:35|
|     1|     50|   5.0|2000-07-30 18:48:51|
|     1|     70|   3.0|2000-07-30 18:40:00|
|     1|    101|   5.0|2000-07-30 18:14:28|
|     1|    110|   4.0|2000-07-30 18:36:16|
|     1|    151|   5.0|2000-07-30 19:07:21|
|     1|    157|   5.0|2000-07-30 19:08:20|
+------+-------+------+-------------------+
only showing top 10 rows



### a. Show the aggregated number of ratings per year

In [34]:
from pyspark.sql.functions import year, col, count
#Agg_ratings_per_year_df = ratings_df.withColumn('YearOfRating',year(col('timestamp'))).groupBy(col('YearOfRating')).count().orderBy(col('YearOfRating'))
Agg_ratings_per_year_df = ratings_df.withColumn('YearOfRating',year(col('timestamp'))).groupBy(col('YearOfRating')).agg(count(col('movieId')).alias('CNT')).orderBy(col('YearOfRating'))

Agg_ratings_per_year_df.show(Agg_ratings_per_year_df.count())

+------------+-----+
|YearOfRating|  CNT|
+------------+-----+
|        1996| 6040|
|        1997| 1916|
|        1998|  507|
|        1999| 2439|
|        2000|10061|
|        2001| 3922|
|        2002| 3478|
|        2003| 4014|
|        2004| 3279|
|        2005| 5813|
|        2006| 4059|
|        2007| 7114|
|        2008| 4351|
|        2009| 4158|
|        2010| 2301|
|        2011| 1690|
|        2012| 4656|
|        2013| 1664|
|        2014| 1439|
|        2015| 6616|
|        2016| 6703|
|        2017| 8198|
|        2018| 6418|
+------------+-----+



### b. Show the average monthly number of ratings

In [48]:
from pyspark.sql.functions import month, avg
Avg_Monthly_Ratings_df = ratings_df.withColumns({'Year': year(col('timestamp')), 'Month': month(col('timestamp'))}) \
                                    .groupBy(col('Year'), col('Month')).count().withColumnRenamed('count','YearMonthRatingCount') \
                                    .select(avg(col('YearMonthRatingCount')))
Avg_Monthly_Ratings_df.show()

+-------------------------+
|avg(YearMonthRatingCount)|
+-------------------------+
|        377.6629213483146|
+-------------------------+



### c. Show the rating levels distribution


In [49]:
Rating_levels_dist_df = ratings_df.groupBy(col('rating')).count().orderBy(col('rating'))
Rating_levels_dist_df.show()

+------+-----+
|rating|count|
+------+-----+
|   0.5| 1370|
|   1.0| 2811|
|   1.5| 1791|
|   2.0| 7551|
|   2.5| 5550|
|   3.0|20047|
|   3.5|13136|
|   4.0|26818|
|   4.5| 8551|
|   5.0|13211|
+------+-----+



### d. Show the 18 movies that are tagged but not rated


In [58]:
Tagged_Not_rated_df = tags_df.alias('t').join(ratings_df.alias('r'), col('t.movieId') == col('r.movieId') , 'left').where(col('r.movieId').isNull())
Tagged_Not_rated_df.show(18)

26/11/14 14:48:49 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------+-------+--------------------+-------------------+------+-------+------+---------+
|userId|movieId|                 tag|          timestamp|userId|movieId|rating|timestamp|
+------+-------+--------------------+-------------------+------+-------+------+---------+
|   288|   7020|      Notable Nudity|2006-06-22 14:54:37|  NULL|   NULL|  NULL|     NULL|
|   318|  30892|           Animation|2009-04-19 16:29:50|  NULL|   NULL|  NULL|     NULL|
|   318|  30892|         Documentary|2009-04-19 16:29:11|  NULL|   NULL|  NULL|     NULL|
|   318|  30892|        Henry Darger|2009-04-19 16:29:20|  NULL|   NULL|  NULL|     NULL|
|   474|   1076|           governess|2006-01-17 18:09:54|  NULL|   NULL|  NULL|     NULL|
|   474|   2939|    In Netflix queue|2006-01-14 01:29:00|  NULL|   NULL|  NULL|     NULL|
|   474|   3338|    In Netflix queue|2006-01-14 01:21:20|  NULL|   NULL|  NULL|     NULL|
|   474|   3456|    In Netflix queue|2006-01-14 01:17:43|  NULL|   NULL|  NULL|     NULL|
|   474|  

### e. Show the movies that have rating but no tag


In [61]:
Rating_No_Tag_df = ratings_df.alias('r').join(tags_df.alias('t'), col('r.movieId') == col('t.movieId'), 'left').where(col('t.movieId').isNull())
Rating_No_Tag_df.show()

26/11/14 14:55:09 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------+-------+------+-------------------+------+-------+----+---------+
|userId|movieId|rating|          timestamp|userId|movieId| tag|timestamp|
+------+-------+------+-------------------+------+-------+----+---------+
|     1|      6|   4.0|2000-07-30 18:37:04|  NULL|   NULL|NULL|     NULL|
|     1|     70|   3.0|2000-07-30 18:40:00|  NULL|   NULL|NULL|     NULL|
|     1|    151|   5.0|2000-07-30 19:07:21|  NULL|   NULL|NULL|     NULL|
|     1|    157|   5.0|2000-07-30 19:08:20|  NULL|   NULL|NULL|     NULL|
|     1|    163|   5.0|2000-07-30 19:00:50|  NULL|   NULL|NULL|     NULL|
|     1|    231|   5.0|2000-07-30 18:19:39|  NULL|   NULL|NULL|     NULL|
|     1|    333|   5.0|2000-07-30 18:19:39|  NULL|   NULL|NULL|     NULL|
|     1|    362|   5.0|2000-07-30 18:43:08|  NULL|   NULL|NULL|     NULL|
|     1|    367|   4.0|2000-07-30 18:28:30|  NULL|   NULL|NULL|     NULL|
|     1|    423|   3.0|2000-07-30 18:39:23|  NULL|   NULL|NULL|     NULL|
|     1|    441|   4.0|2000-07-30 18:1

### f.Focusing on the rated untagged movies with more than 30 user ratings, show the top 10 movies in terms of average rating and number of ratings

In [69]:
from pyspark.sql.functions import count, avg
Movies_Rates30_df = ratings_df.groupBy('movieId').agg(avg('rating').alias('RatingAvg'), count('movieId').alias('count')).where(col('count') > 30)
Rating30_No_Tag_df = Movies_Rates30_df.alias('r').join(tags_df.alias('t'), col('r.movieId') == col('t.movieId'), 'left').where(col('t.movieId').isNull())
Rating30_No_Tag_df.orderBy(col('RatingAvg').desc() , col('count').desc()).show(10)

26/11/14 15:07:54 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+-------+-----------------+-----+------+-------+----+---------+
|movieId|        RatingAvg|count|userId|movieId| tag|timestamp|
+-------+-----------------+-----+------+-------+----+---------+
|   3275| 4.22093023255814|   43|  NULL|   NULL|NULL|     NULL|
|   1199|4.177966101694915|   59|  NULL|   NULL|NULL|     NULL|
|   1172|4.161764705882353|   34|  NULL|   NULL|NULL|     NULL|
|   4011|4.155913978494624|   93|  NULL|   NULL|NULL|     NULL|
|   3681|4.151515151515151|   33|  NULL|   NULL|NULL|     NULL|
|  44555|4.117647058823529|   34|  NULL|   NULL|NULL|     NULL|
|  78499|4.109090909090909|   55|  NULL|   NULL|NULL|     NULL|
|   1673|4.076923076923077|   39|  NULL|   NULL|NULL|     NULL|
|   2858|4.056372549019608|  204|  NULL|   NULL|NULL|     NULL|
|   2542|4.052238805970149|   67|  NULL|   NULL|NULL|     NULL|
+-------+-----------------+-----+------+-------+----+---------+
only showing top 10 rows



###  g. What is the average number of tags per movie in tagsDF? And the average number of tags per user? How does it compare with the average number of tags a user assigns to a movie?

In [76]:
AM = tags_df.groupBy('movieId').count()
AU = tags_df.groupBy('userId').count()
AMU = tags_df.groupBy('movieId','userId').count()
Comparison_df = AMU.join(AM, AM.movieId == AMU.movieId ,'left').join(AU, AMU.userId == AU.userId, 'left')
Comparison_df.show(10)

26/11/14 15:27:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/11/14 15:27:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+-------+------+-----+-------+-----+------+-----+
|movieId|userId|count|movieId|count|userId|count|
+-------+------+-----+-------+-----+------+-----+
|   2959|   435|    1|   2959|   54|   435|    4|
|   1917|   474|    1|   1917|    1|   474| 1507|
|    970|   474|    1|    970|    1|   474| 1507|
|   4621|   474|    1|   4621|    1|   474| 1507|
|   1343|   474|    1|   1343|    5|   474| 1507|
|   2324|   474|    1|   2324|    8|   474| 1507|
|   7981|   474|    1|   7981|    1|   474| 1507|
|  97304|   537|    3|  97304|    3|   537|  100|
|   1350|   474|    1|   1350|    1|   474| 1507|
|   2059|   474|    1|   2059|    1|   474| 1507|
+-------+------+-----+-------+-----+------+-----+
only showing top 10 rows



### h.Identify the users that tagged movies without rating them

In [81]:
tags_df.join(ratings_df, (tags_df.movieId == ratings_df.movieId) & (tags_df.userId == ratings_df.userId) , 'left').where(ratings_df.rating.isNull()).show(10)


26/11/14 15:36:09 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------+-------+-----------------+-------------------+------+-------+------+---------+
|userId|movieId|              tag|          timestamp|userId|movieId|rating|timestamp|
+------+-------+-----------------+-------------------+------+-------+------+---------+
|    21|   1569|  romantic comedy|2014-12-28 22:23:33|  NULL|   NULL|  NULL|     NULL|
|    21|   1569|          wedding|2014-12-28 22:23:39|  NULL|   NULL|  NULL|     NULL|
|    21| 118985|          painter|2014-12-28 22:24:37|  NULL|   NULL|  NULL|     NULL|
|   119|    260|          classic|2015-07-03 16:55:30|  NULL|   NULL|  NULL|     NULL|
|   119|    260|             Nerd|2015-07-03 16:55:20|  NULL|   NULL|  NULL|     NULL|
|   193|   7361|alternate reality|2015-07-02 17:12:48|  NULL|   NULL|  NULL|     NULL|
|   193|   7361|           memory|2015-07-02 17:12:51|  NULL|   NULL|  NULL|     NULL|
|   193|   7361|thought-provoking|2015-07-02 17:12:43|  NULL|   NULL|  NULL|     NULL|
|   288|   7020|   Notable Nudity|2006-06-2

### i. What is the average number of ratings per user in ratings DF? And the average number of ratings per movie?

In [84]:
Avg_ratePerUser = ratings_df.groupBy('userId').count().agg(avg('count'))
Avg_ratePerMovie = ratings_df.groupBy('movieId').count().agg(avg('count'))
Avg_ratePerUser.show()
Avg_ratePerMovie.show()

+------------------+
|        avg(count)|
+------------------+
|165.30491803278687|
+------------------+

+------------------+
|        avg(count)|
+------------------+
|10.369806663924312|
+------------------+



### j. What is the predominant (frequency based) genre per rating level? 

In [95]:
## the answer could be wrong. I forgot there is another dataset called movies where we have genres column,. Hence, I considered movies as genres.
from pyspark.sql import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy('rating').orderBy(col('count').desc())
Grouped_df = ratings_df.groupBy('movieId', 'rating').count()
Grouped_df.withColumn('RN', row_number().over(windowSpec)).where(col('RN') == 1).show()

+-------+------+-----+---+
|movieId|rating|count| RN|
+-------+------+-----+---+
|   2683|   0.5|    7|  1|
|   2571|   0.5|    5|  2|
|   2617|   0.5|    5|  3|
|   1732|   0.5|    5|  4|
|   1644|   0.5|    5|  5|
|    435|   0.5|    4|  6|
|    788|   0.5|    4|  7|
|   6934|   0.5|    4|  8|
|   8957|   0.5|    4|  9|
|   3593|   0.5|    4| 10|
|   4310|   0.5|    4| 11|
|   3949|   0.5|    4| 12|
|   3977|   0.5|    4| 13|
|   1407|   0.5|    4| 14|
|   1101|   0.5|    4| 15|
|   1760|   0.5|    4| 16|
|   5313|   0.5|    4| 17|
|   2720|   0.5|    4| 18|
|    317|   0.5|    4| 19|
|    186|   0.5|    4| 20|
+-------+------+-----+---+
only showing top 20 rows



### k. What is the predominant tag per genre and the most tagged genres?

In [97]:
## the answer could be wrong. I forgot there is another dataset called movies where we have genres column,. Hence, I considered movies as genres.

window_spec2 = Window.partitionBy('tag').orderBy(col('count').desc())

Groupedtags = tags_df.groupBy('movieId', 'tag').count()

Groupedtags.withColumn('RN', row_number().over(window_spec2)).where(col('RN') == 1).show()

+-------+--------------------+-----+---+
|movieId|                 tag|count| RN|
+-------+--------------------+-----+---+
|   4552|         """artsy"""|    1|  1|
|  37729|06 Oscar Nominate...|    1|  1|
|    918|               1900s|    1|  1|
|   1084|               1920s|    1|  1|
|   2664|               1950s|    1|  1|
|   1093|               1960s|    1|  1|
|   3556|               1970s|    1|  1|
|   2145|               1980s|    1|  1|
|    296|               1990s|    1|  1|
|  68237|           2001-like|    1|  1|
|  62336|        2D animation|    1|  1|
|   5694|                70mm|    1|  1|
| 156371|                80's|    1|  1|
|   2071|                AIDs|    1|  1|
|   5527|            AS Byatt|    1|  1|
|    296|             AWESOME|    1|  1|
|    745|             Aardman|    1|  1|
|   2919|Academy award (Be...|    1|  1|
|   1240|              Action|    1|  1|
|   2694|        Adam Sandler|    1|  1|
+-------+--------------------+-----+---+
only showing top

#### l. What are the most predominant (popularity based) movies?


In [98]:
ratings_df.groupBy("movieId").count().orderBy(col('count').desc()).show()

+-------+-----+
|movieId|count|
+-------+-----+
|    356|  329|
|    318|  317|
|    296|  307|
|    593|  279|
|   2571|  278|
|    260|  251|
|    480|  238|
|    110|  237|
|    589|  224|
|    527|  220|
|   2959|  218|
|      1|  215|
|   1196|  211|
|     50|  204|
|   2858|  204|
|     47|  203|
|    780|  202|
|    150|  201|
|   1198|  200|
|   4993|  198|
+-------+-----+
only showing top 20 rows



### m. Top 10 movies in terms of average rating (provided more than 30 users reviewed them)